# 1. NeuroGolf 2026 EDA: ARC Task Structure, Shapes, and Color Patterns

This notebook is designed to run inside Kaggle for the NeuroGolf 2026 competition. It focuses on understanding the ARC-style task files before modeling: input/output grid sizes, color usage, train/test pair counts, and reusable visual inspection tools. The plotting palette is standardized to `viridis` for continuous charts, while ARC grid visualizations use the canonical 0-9 ARC color map so the puzzle colors remain interpretable.

## 1.1 Setup: Imports and Display Defaults

Kaggle already provides the notebook runtime, so this cell only imports common packages and sets consistent plotting defaults. No local dependency setup is required.

In [ ]:
from __future__ import annotations

import json
import math
import os
from collections import Counter
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap, BoundaryNorm

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

PLOT_CMAP = "viridis"
plt.style.use("default")
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Canonical ARC palette. Keep this separate from viridis because ARC colors are semantic puzzle tokens.
ARC_COLORS = [
    "#000000",  # 0 black
    "#0074D9",  # 1 blue
    "#FF4136",  # 2 red
    "#2ECC40",  # 3 green
    "#FFDC00",  # 4 yellow
    "#AAAAAA",  # 5 gray
    "#F012BE",  # 6 magenta
    "#FF851B",  # 7 orange
    "#7FDBFF",  # 8 light blue
    "#870C25",  # 9 maroon
]
ARC_CMAP = ListedColormap(ARC_COLORS)
ARC_NORM = BoundaryNorm(np.arange(-0.5, 10.5, 1), ARC_CMAP.N)

## 1.2 Data Discovery: Locate Kaggle Inputs Without Hard-Coding

The competition data may appear under a Kaggle input directory name such as `neurogolf-2026` or a public-data companion dataset. This scanner finds JSON task files under `/kaggle/input` when running on Kaggle, and falls back to local relative folders when developing elsewhere.

In [ ]:
KAGGLE_INPUT = Path("/kaggle/input")
LOCAL_CANDIDATES = [Path("../input"), Path("input"), Path("data"), Path("../data")]

def candidate_roots() -> list[Path]:
    roots: list[Path] = []
    if KAGGLE_INPUT.exists():
        roots.extend([p for p in KAGGLE_INPUT.iterdir() if p.is_dir()])
        roots.append(KAGGLE_INPUT)
    roots.extend([p for p in LOCAL_CANDIDATES if p.exists()])
    return roots

def find_json_files() -> list[Path]:
    files: list[Path] = []
    for root in candidate_roots():
        files.extend(root.rglob("*.json"))
    return sorted(set(files))

json_files = find_json_files()
print(f"Found {len(json_files):,} JSON files")
for path in json_files[:25]:
    print(path)
if len(json_files) > 25:
    print(f"... {len(json_files) - 25:,} more")

## 1.3 Data Loading: Normalize `task001.json` and `all_tasks.json` Formats

NeuroGolf submissions are one ONNX file per task, named `task001.onnx` through `task400.onnx`. Public data can be distributed either as one JSON file per task or as a combined dictionary. This loader handles both shapes and returns a single `tasks` mapping keyed by task id.

In [ ]:
def is_task_payload(obj: Any) -> bool:
    return isinstance(obj, dict) and "train" in obj and "test" in obj

def normalize_task_id(path: Path, key: str | None = None) -> str:
    if key:
        key = str(key)
        return key[:-5] if key.endswith(".json") else key
    return path.stem

def load_tasks(files: list[Path]) -> dict[str, dict[str, Any]]:
    tasks: dict[str, dict[str, Any]] = {}
    for path in files:
        try:
            with path.open("r") as f:
                obj = json.load(f)
        except Exception as exc:
            print(f"Skipping {path}: {exc}")
            continue

        if is_task_payload(obj):
            tasks[normalize_task_id(path)] = obj
        elif isinstance(obj, dict):
            for key, value in obj.items():
                if is_task_payload(value):
                    tasks[normalize_task_id(path, key)] = value
        elif isinstance(obj, list):
            for idx, value in enumerate(obj, start=1):
                if is_task_payload(value):
                    tasks[f"task{idx:03d}"] = value
    return dict(sorted(tasks.items()))

tasks = load_tasks(json_files)
print(f"Loaded {len(tasks):,} tasks")
list(tasks)[:10]

## 2.1 Task Inventory: Pair Counts, Grid Sizes, and Color Sets

This table converts nested ARC examples into task-level summary features. It is intentionally broad rather than clever: these columns make it easy to spot unusual tasks before deciding which solver families deserve attention.

In [ ]:
def grid_shape(grid: list[list[int]]) -> tuple[int, int]:
    arr = np.asarray(grid)
    return tuple(arr.shape) if arr.ndim == 2 else (0, 0)

def grid_colors(grid: list[list[int]]) -> set[int]:
    arr = np.asarray(grid)
    return set(map(int, np.unique(arr))) if arr.size else set()

def task_summary(task_id: str, task: dict[str, Any]) -> dict[str, Any]:
    train = task.get("train", [])
    test = task.get("test", [])
    pairs = train + test

    input_shapes = [grid_shape(pair["input"]) for pair in pairs if "input" in pair]
    output_shapes = [grid_shape(pair["output"]) for pair in pairs if "output" in pair]
    train_output_shapes = [grid_shape(pair["output"]) for pair in train if "output" in pair]

    input_colors = set().union(*(grid_colors(pair["input"]) for pair in pairs if "input" in pair)) if pairs else set()
    output_colors = set().union(*(grid_colors(pair["output"]) for pair in pairs if "output" in pair)) if output_shapes else set()

    train_shape_changes = [
        grid_shape(pair["input"]) != grid_shape(pair["output"])
        for pair in train
        if "input" in pair and "output" in pair
    ]

    return {
        "task_id": task_id,
        "n_train": len(train),
        "n_test": len(test),
        "has_test_outputs": all("output" in pair for pair in test) if test else False,
        "input_shapes": sorted(set(input_shapes)),
        "output_shapes": sorted(set(output_shapes)),
        "train_output_shapes": sorted(set(train_output_shapes)),
        "n_input_colors": len(input_colors),
        "n_output_colors": len(output_colors),
        "input_colors": tuple(sorted(input_colors)),
        "output_colors": tuple(sorted(output_colors)),
        "shape_changes_in_train": any(train_shape_changes),
        "max_input_area": max((r * c for r, c in input_shapes), default=0),
        "max_output_area": max((r * c for r, c in output_shapes), default=0),
    }

summary_df = pd.DataFrame([task_summary(task_id, task) for task_id, task in tasks.items()])
summary_df.head(10)

## 2.2 Inventory Checks: Confirm Expected Task Coverage

The competition summary describes up to 400 submitted ONNX files. This cell checks whether the discovered public data aligns with that shape and highlights missing task ids if task ids follow the `task001` convention.

In [ ]:
print(summary_df.shape)
if summary_df.empty:
    print("No tasks loaded yet. Attach the Kaggle competition/public data and rerun from the top.")
else:
    display(summary_df.describe(include="all"))

    expected_ids = {f"task{i:03d}" for i in range(1, 401)}
    observed_ids = set(summary_df["task_id"])
    missing_expected = sorted(expected_ids - observed_ids)
    extra_ids = sorted(observed_ids - expected_ids)

    print(f"Expected-style tasks present: {len(expected_ids & observed_ids):,} / 400")
    print(f"Missing expected ids: {missing_expected[:20]}{' ...' if len(missing_expected) > 20 else ''}")
    print(f"Non-standard ids: {extra_ids[:20]}{' ...' if len(extra_ids) > 20 else ''}")

## 3.1 Pair Distributions: Training Examples and Test Cases

Pair count distributions are a quick proxy for how much evidence each task gives us. Low-example tasks tend to need stronger priors or simpler transformation hypotheses.

In [ ]:
if not summary_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    summary_df["n_train"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color=plt.get_cmap(PLOT_CMAP)(0.62))
    axes[0].set_title("Training examples per task")
    axes[0].set_xlabel("n_train")
    axes[0].set_ylabel("task count")

    summary_df["n_test"].value_counts().sort_index().plot(kind="bar", ax=axes[1], color=plt.get_cmap(PLOT_CMAP)(0.82))
    axes[1].set_title("Test cases per task")
    axes[1].set_xlabel("n_test")
    axes[1].set_ylabel("task count")
    plt.tight_layout()

## 3.2 Grid Geometry: Input and Output Areas

NeuroGolf rewards small, efficient ONNX networks, so grid geometry matters. Larger input/output areas can increase memory and operation costs, while shape-changing tasks often need different solver templates than same-shape recoloring tasks.

In [ ]:
if not summary_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    summary_df["max_input_area"].hist(ax=axes[0], bins=30, color=plt.get_cmap(PLOT_CMAP)(0.50))
    axes[0].set_title("Max input area")
    axes[0].set_xlabel("cells")
    axes[0].set_ylabel("task count")

    summary_df["max_output_area"].hist(ax=axes[1], bins=30, color=plt.get_cmap(PLOT_CMAP)(0.70))
    axes[1].set_title("Max output area")
    axes[1].set_xlabel("cells")

    summary_df["shape_changes_in_train"].value_counts().sort_index().plot(kind="bar", ax=axes[2], color=[plt.get_cmap(PLOT_CMAP)(0.35), plt.get_cmap(PLOT_CMAP)(0.85)])
    axes[2].set_title("Shape changes in training pairs")
    axes[2].set_xlabel("shape changes")
    axes[2].set_ylabel("task count")
    plt.tight_layout()

if not summary_df.empty:
    display(summary_df.sort_values(["max_input_area", "max_output_area"], ascending=False).head(15))

## 3.3 Color Usage: Palette Size and Token Frequency

ARC colors are discrete symbols, not natural-image colors. These counts help identify tasks that may be solved by color mapping, object extraction, masking, or background manipulation.

In [ ]:
def count_grid_values(tasks: dict[str, dict[str, Any]], split: str, field: str) -> Counter:
    counts: Counter = Counter()
    for task in tasks.values():
        for pair in task.get(split, []):
            if field in pair:
                counts.update(np.asarray(pair[field]).ravel().astype(int).tolist())
    return counts

train_input_color_counts = count_grid_values(tasks, "train", "input")
train_output_color_counts = count_grid_values(tasks, "train", "output")

color_df = pd.DataFrame({
    "color": range(10),
    "train_input_cells": [train_input_color_counts.get(i, 0) for i in range(10)],
    "train_output_cells": [train_output_color_counts.get(i, 0) for i in range(10)],
})
display(color_df)

if not color_df.empty:
    ax = color_df.set_index("color")[["train_input_cells", "train_output_cells"]].plot(kind="bar", figsize=(10, 4), colormap=PLOT_CMAP)
    ax.set_title("ARC token frequency in training pairs")
    ax.set_xlabel("ARC color token")
    ax.set_ylabel("cell count")
    plt.tight_layout()

if not summary_df.empty:
    ax = summary_df["n_input_colors"].value_counts().sort_index().plot(kind="bar", figsize=(8, 4), color=plt.get_cmap(PLOT_CMAP)(0.68))
    ax.set_title("Unique input colors per task")
    ax.set_xlabel("unique colors")
    ax.set_ylabel("task count")
    plt.tight_layout()

## 4.1 Visual Tools: Inspect Any ARC Task by Id

Manual inspection is still the fastest way to understand ARC transformations. The helper below renders train and test examples in a compact grid, with missing test outputs handled gracefully.

In [ ]:
def show_grid(ax: plt.Axes, grid: list[list[int]], title: str) -> None:
    arr = np.asarray(grid)
    ax.imshow(arr, cmap=ARC_CMAP, norm=ARC_NORM)
    ax.set_title(title, fontsize=10)
    ax.set_xticks(np.arange(-0.5, arr.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, arr.shape[0], 1), minor=True)
    ax.grid(which="minor", color="#555555", linewidth=0.5, alpha=0.55)
    ax.tick_params(which="both", left=False, bottom=False, labelleft=False, labelbottom=False)

def show_task(task_id: str, max_pairs: int | None = None) -> None:
    task = tasks[task_id]
    rows: list[tuple[str, int, dict[str, Any]]] = []
    for split in ["train", "test"]:
        pairs = task.get(split, [])
        if max_pairs is not None:
            pairs = pairs[:max_pairs]
        rows.extend((split, idx, pair) for idx, pair in enumerate(pairs, start=1))

    n_rows = len(rows)
    n_cols = 2
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6, max(2.2, 2.4 * n_rows)))
    axes = np.asarray(axes).reshape(n_rows, n_cols)
    fig.suptitle(task_id, fontsize=14, y=1.0)

    for row_idx, (split, idx, pair) in enumerate(rows):
        show_grid(axes[row_idx, 0], pair["input"], f"{split} {idx} input")
        if "output" in pair:
            show_grid(axes[row_idx, 1], pair["output"], f"{split} {idx} output")
        else:
            axes[row_idx, 1].axis("off")
            axes[row_idx, 1].set_title(f"{split} {idx} output unavailable", fontsize=10)
    plt.tight_layout()

if tasks:
    first_task_id = next(iter(tasks))
    show_task(first_task_id)

## 4.2 Visual Sampling: Review Diverse Tasks by Simple Heuristics

These curated samples are not model logic; they are a way to quickly browse different task types: largest grids, many-color tasks, shape-changing tasks, and compact same-shape tasks.

In [ ]:
sample_ids: list[str] = []
if not summary_df.empty:
    sample_ids.extend(summary_df.sort_values("max_input_area", ascending=False).head(2)["task_id"].tolist())
    sample_ids.extend(summary_df.sort_values("n_input_colors", ascending=False).head(2)["task_id"].tolist())
    sample_ids.extend(summary_df.query("shape_changes_in_train == True").head(2)["task_id"].tolist())
    sample_ids.extend(summary_df.query("shape_changes_in_train == False").sort_values("max_input_area").head(2)["task_id"].tolist())

sample_ids = list(dict.fromkeys(sample_ids))
sample_ids

In [ ]:
for task_id in sample_ids[:8]:
    show_task(task_id, max_pairs=3)

## 5.1 Solver Planning Signals: Candidate Buckets for Later Notebooks

This first-pass bucketing creates a lightweight bridge from EDA to modeling. Later notebooks can replace these rough labels with real solver diagnostics and ONNX export checks.

In [ ]:
def assign_bucket(row: pd.Series) -> str:
    if row["shape_changes_in_train"]:
        return "shape-changing"
    if row["n_input_colors"] <= 2:
        return "low-color same-shape"
    if row["max_input_area"] <= 100:
        return "small same-shape"
    return "larger same-shape"

if not summary_df.empty:
    summary_df = summary_df.copy()
    summary_df["eda_bucket"] = summary_df.apply(assign_bucket, axis=1)
    display(summary_df["eda_bucket"].value_counts().rename_axis("bucket").reset_index(name="task_count"))

    ax = summary_df["eda_bucket"].value_counts().plot(kind="bar", figsize=(9, 4), color=plt.get_cmap(PLOT_CMAP)(0.74))
    ax.set_title("EDA task buckets for solver planning")
    ax.set_xlabel("bucket")
    ax.set_ylabel("task count")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()

summary_df.head()

## 5.2 Export EDA Tables: Keep Lightweight Artifacts for Follow-Up Notebooks

On Kaggle this writes compact CSV summaries under `/kaggle/working`, which makes them easy to attach to later notebooks or inspect after a run.

In [ ]:
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
if not summary_df.empty:
    summary_path = OUTPUT_DIR / "neurogolf_eda_task_summary.csv"
    color_path = OUTPUT_DIR / "neurogolf_eda_color_counts.csv"
    summary_df.to_csv(summary_path, index=False)
    color_df.to_csv(color_path, index=False)
    print(f"Wrote {summary_path}")
    print(f"Wrote {color_path}")